# **Fine-Tuning BERT for POS Tagging & Chunking**

**STEP 1: Install Libraries**

In [ ]:
!pip install transformers datasets seqeval

In [ ]:
!pip install datasets==2.19.0

**STEP 2: Load Dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

**STEP 3: Get Labels**

In [ ]:
label_list = dataset["train"].features["pos_tags"].feature.names
num_labels = len(label_list)

print(label_list)

In [ ]:
label_list = dataset["train"].features["chunk_tags"].feature.names
num_labels = len(label_list)

print(label_list)

**STEP 4: Load Tokenizer**

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["pos_tags"]):   # ✅ CHANGE HERE
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

**STEP 5: Apply Preprocessing**

In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

**STEP 6: Load Model**

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=num_labels
)

**STEP 7: Training Setup**

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01
)

In [ ]:
from datasets import load_metric

metric = load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

**STEP 8: Trainer**

In [ ]:
from transformers import DataCollatorForTokenClassification

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,   # ✅ THIS LINE FIXES ERROR
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

In [ ]:
sentence = "John works at Google in California"

inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs)

predictions = outputs.logits.argmax(dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, pred in zip(tokens, predictions[0]):
    print(token, label_list[pred])